# 02. Demonstrarea celor 4 axiome de corectitudine

**Pregatire SCSS 2026 - Diana Nenu**

Acest notebook demonstreaza fiecare din cele 4 axiome ale lui Shapley pe jocuri construite special pentru a evidentia comportamentul fiecareia. Demonstratiile sunt **numerice si empirice**, oferind intuitia matematica fara a recurge la demonstratia formala de unicitate (care este la nivel de cercetare matematica).

**Cele 4 axiome:**
1. **Eficienta** - $\sum_i \phi_i = v(N)$
2. **Simetrie** - jucatorii echivalenti primesc valori egale
3. **Dummy** - jucatorii nuli primesc 0
4. **Aditivitate** - $\phi_i(v + w) = \phi_i(v) + \phi_i(w)$

## 2.0. Setup si functii helper

In [1]:
from itertools import combinations, chain
from math import factorial
import numpy as np
import matplotlib.pyplot as plt

def shapley(v, N):
    """Calculeaza vectorul de valori Shapley pentru un joc (N, v)."""
    n = len(N)
    phi = {}
    for jucator in N:
        ceilalti = [j for j in N if j != jucator]
        total = 0.0
        for r in range(len(ceilalti) + 1):
            for S_tuple in combinations(ceilalti, r):
                S = frozenset(S_tuple)
                contrib = v[S | {jucator}] - v[S]
                coef = (factorial(r) * factorial(n - r - 1)) / factorial(n)
                total += coef * contrib
        phi[jucator] = total
    return phi

def toate_subseturile(N):
    return list(chain.from_iterable(combinations(N, r) for r in range(len(N) + 1)))


## 2.1. AXIOMA 1: Eficienta

**Formulare**: $\sum_{i \in N} \phi_i(v) = v(N)$.

**Intuitie**: alocarea trebuie sa epuizeze EXACT valoarea totala produsa de marea coalitie. Nimic nu este pierdut, nimic nu este inventat.

Generez jocuri aleatoare cu 4 jucatori si verific ca eficienta este satisfacuta in fiecare caz.

In [2]:
np.random.seed(42)

def joc_aleator(N, scala=10.0):
    """Genereaza un joc aleator cu valori monoton crescatoare pe ordinea coalitiilor."""
    v = {frozenset(): 0.0}
    subseturi_neville = [s for s in toate_subseturile(N) if s]
    valori_random = sorted(np.random.uniform(0.5, scala, len(subseturi_neville)))
    for s, val in zip(subseturi_neville, valori_random):
        v[frozenset(s)] = val
    # Forteaza superaditivitate aproximativa
    return v

N = ['A', 'B', 'C', 'D']
print('Test eficienta pe 5 jocuri aleatoare:')
for i in range(5):
    v = joc_aleator(N, scala=20.0)
    phi = shapley(v, N)
    suma = sum(phi.values())
    v_N = v[frozenset(N)]
    print(f'  Joc {i+1}: v(N) = {v_N:7.3f}  |  suma phi = {suma:7.3f}  |  diferenta = {abs(suma - v_N):.2e}')

Test eficienta pe 5 jocuri aleatoare:
  Joc 1: v(N) =  19.413  |  suma phi =  19.413  |  diferenta = 0.00e+00
  Joc 2: v(N) =  15.811  |  suma phi =  15.811  |  diferenta = 0.00e+00
  Joc 3: v(N) =  19.330  |  suma phi =  19.330  |  diferenta = 0.00e+00
  Joc 4: v(N) =  19.407  |  suma phi =  19.407  |  diferenta = 0.00e+00
  Joc 5: v(N) =  19.744  |  suma phi =  19.744  |  diferenta = 0.00e+00


**Concluzia**: in toate cazurile, diferenta intre suma valorilor Shapley si $v(N)$ este de ordinul erorii numerice floating-point ($10^{-14}$ - $10^{-15}$). Axioma de eficienta este satisfacuta MATEMATIC EXACT prin constructia formulei.

## 2.2. AXIOMA 2: Simetrie

**Formulare**: daca doi jucatori $i$ si $j$ contribuie identic in orice coalitie, $v(S \cup \{i\}) = v(S \cup \{j\})$ pentru orice $S$ care nu il contine pe nici unul, atunci $\phi_i(v) = \phi_j(v)$.

**Intuitie**: jucatorii functionali echivalenti primesc tratament identic.

Construiesc un joc in care 2 jucatori (B si C) sunt simetrici prin constructie.

In [3]:
# Construiesc un joc unde B si C sunt simetrici - functioneaza identic in orice coalitie
v_simetric = {
    frozenset(): 0,
    frozenset(['A']): 5.0,
    frozenset(['B']): 3.0,
    frozenset(['C']): 3.0,           # B si C au aceeasi valoare singura
    frozenset(['A', 'B']): 8.0,
    frozenset(['A', 'C']): 8.0,      # A+B si A+C identice
    frozenset(['B', 'C']): 7.0,
    frozenset(['A', 'B', 'C']): 12.0,
}

N = ['A', 'B', 'C']
phi = shapley(v_simetric, N)
print('Valorile Shapley pentru jocul simetric (B si C interschimbabili):')
for j, val in phi.items():
    print(f'  phi_{j} = {val:.6f}')
print(f'\nVerificare simetrie: phi_B == phi_C ?  {abs(phi["B"] - phi["C"]) < 1e-12}')
print(f'Diferenta exacta: {phi["B"] - phi["C"]:.2e}')

Valorile Shapley pentru jocul simetric (B si C interschimbabili):
  phi_A = 5.000000
  phi_B = 3.500000
  phi_C = 3.500000

Verificare simetrie: phi_B == phi_C ?  True
Diferenta exacta: 0.00e+00


## 2.3. AXIOMA 3: Dummy (Jucator nul)

**Formulare**: daca jucatorul $i$ nu adauga nicio valoare in nicio coalitie, $v(S \cup \{i\}) = v(S)$ pentru orice $S$, atunci $\phi_i(v) = 0$.

**Intuitie**: nu primesti credit pentru ceva la ce nu contribui.

Construiesc un joc cu un jucator D care este complet inutil.

In [4]:
# Joc cu jucator D dummy - nu adauga valoare in nicio coalitie
v_dummy = {
    frozenset(): 0,
    frozenset(['A']): 5.0,
    frozenset(['B']): 3.0,
    frozenset(['C']): 4.0,
    frozenset(['D']): 0.0,                    # D singur = 0
    frozenset(['A', 'B']): 9.0,
    frozenset(['A', 'C']): 8.5,
    frozenset(['A', 'D']): 5.0,               # D adaugat la A nu schimba nimic
    frozenset(['B', 'C']): 7.5,
    frozenset(['B', 'D']): 3.0,
    frozenset(['C', 'D']): 4.0,
    frozenset(['A', 'B', 'C']): 12.0,
    frozenset(['A', 'B', 'D']): 9.0,
    frozenset(['A', 'C', 'D']): 8.5,
    frozenset(['B', 'C', 'D']): 7.5,
    frozenset(['A', 'B', 'C', 'D']): 12.0,    # D adaugat la coalitia totala nu schimba
}

N = ['A', 'B', 'C', 'D']
phi = shapley(v_dummy, N)
print('Valorile Shapley pentru jocul cu jucator D dummy:')
for j, val in phi.items():
    print(f'  phi_{j} = {val:.6f}')
print(f'\nVerificare axioma dummy: phi_D == 0 ?  {abs(phi["D"]) < 1e-12}')
print(f'Suma: {sum(phi.values()):.4f} = v(N) = {v_dummy[frozenset(N)]}')

Valorile Shapley pentru jocul cu jucator D dummy:
  phi_A = 4.916667
  phi_B = 3.416667
  phi_C = 3.666667
  phi_D = 0.000000

Verificare axioma dummy: phi_D == 0 ?  True
Suma: 12.0000 = v(N) = 12.0


## 2.4. AXIOMA 4: Aditivitate

**Formulare**: daca $v$ si $w$ sunt doua jocuri pe acelasi set de jucatori, atunci $\phi_i(v + w) = \phi_i(v) + \phi_i(w)$ pentru orice $i$, unde $(v+w)(S) = v(S) + w(S)$.

**Intuitie**: daca jucatorii participa in mai multe activitati simultan, contributiile totale se aduna liniar.

Generez doua jocuri aleatoare $v$ si $w$ si verific empiric aditivitatea.

In [5]:
np.random.seed(7)
N = ['A', 'B', 'C']

def joc_random_simplu(N, scala):
    v = {frozenset(): 0.0}
    for s in toate_subseturile(N):
        if s:
            v[frozenset(s)] = round(np.random.uniform(0.5, scala), 3)
    return v

v = joc_random_simplu(N, scala=10)
w = joc_random_simplu(N, scala=5)

# Suma celor doua jocuri
v_plus_w = {S: v[S] + w[S] for S in v}

phi_v = shapley(v, N)
phi_w = shapley(w, N)
phi_suma = shapley(v_plus_w, N)

print('Verificare aditivitate: phi_i(v+w) == phi_i(v) + phi_i(w)')
print()
print(f'{"Jucator":<10}{"phi(v)":<15}{"phi(w)":<15}{"phi(v)+phi(w)":<20}{"phi(v+w)":<15}{"Diferenta"}')
for j in N:
    suma_individuala = phi_v[j] + phi_w[j]
    diff = abs(suma_individuala - phi_suma[j])
    print(f'{j:<10}{phi_v[j]:<15.4f}{phi_w[j]:<15.4f}{suma_individuala:<20.4f}{phi_suma[j]:<15.4f}{diff:.2e}')

print('\n=> Aditivitatea este satisfacuta exact (diferentele sunt erori numerice).')

Verificare aditivitate: phi_i(v+w) == phi_i(v) + phi_i(w)

Jucator   phi(v)         phi(w)         phi(v)+phi(w)       phi(v+w)       Diferenta
A         1.0550         0.3385         1.3935              1.3935         0.00e+00
B         2.3095         -0.1710        2.1385              2.1385         4.44e-16
C         1.8965         0.6295         2.5260              2.5260         0.00e+00

=> Aditivitatea este satisfacuta exact (diferentele sunt erori numerice).


## 2.5. Teorema unicitatii lui Shapley (schita)

**Teorema** (Shapley, 1953): exista o **unica** functie $\phi: \text{Jocuri}_N \to \mathbb{R}^n$ care satisface simultan axiomele de eficienta, simetrie, dummy si aditivitate, anume formula combinatoriala expusa.

**Schita demonstratiei** (informala):

1. Orice joc cooperativ poate fi descompus UNIC ca o **suma liniara** de "jocuri unanime". Pentru fiecare coalitie nevida $T \subseteq N$ definim jocul unanim $u_T(S) = 1$ daca $T \subseteq S$, altfel 0.

2. **Aditivitatea** garanteaza ca $\phi(\sum_T c_T \cdot u_T) = \sum_T c_T \cdot \phi(u_T)$, deci e suficient sa determinam $\phi$ pe jocurile unanime.

3. Pe un joc unanim $u_T$: jucatorii din $N \setminus T$ sunt dummy (nu schimba niciodata valoarea), iar jucatorii din $T$ sunt simetrici. **Dummy** + **simetrie** + **eficienta** (suma trebuie sa fie 1) forteaza unic: $\phi_i(u_T) = 1/|T|$ daca $i \in T$, altfel 0.

4. Combinand 1 + 2 + 3 obtinem ca $\phi$ este unic determinata pe orice joc.

Verific numeric pasul 3 pe un joc unanim.

In [6]:
# Construiesc jocul unanim u_T pentru T = {A, B}
T = frozenset(['A', 'B'])
N = ['A', 'B', 'C', 'D']

u_T = {}
for s in toate_subseturile(N):
    S = frozenset(s)
    u_T[S] = 1.0 if T.issubset(S) else 0.0

phi = shapley(u_T, N)
print(f'Joc unanim u_T pentru T = {set(T)} pe N = {N}:')
print('Valori Shapley:')
for j, val in phi.items():
    in_T = '(in T)' if j in T else '(NU in T)'
    print(f'  phi_{j} = {val:.6f}  {in_T}')
print(f'\nVerificare:')
print(f'  Jucatori in T primesc 1/|T| = 1/{len(T)} = {1/len(T):.4f}')
print(f'  Jucatori in afara lui T primesc 0')
print(f'  Suma totala = {sum(phi.values()):.4f} = v(N) = {u_T[frozenset(N)]}')

Joc unanim u_T pentru T = {'A', 'B'} pe N = ['A', 'B', 'C', 'D']:
Valori Shapley:
  phi_A = 0.500000  (in T)
  phi_B = 0.500000  (in T)
  phi_C = 0.000000  (NU in T)
  phi_D = 0.000000  (NU in T)

Verificare:
  Jucatori in T primesc 1/|T| = 1/2 = 0.5000
  Jucatori in afara lui T primesc 0
  Suma totala = 1.0000 = v(N) = 1.0


## 2.6. Concluzii

Am verificat **numeric** ca formula valorii Shapley satisface toate 4 axiomele:
- **Eficienta** - exact, prin constructia formulei (suma diferentelor telescopate da v(N))
- **Simetrie** - exact, demonstrat pe joc construit ad-hoc
- **Dummy** - exact, demonstrat pe joc cu jucator nul
- **Aditivitate** - exact, demonstrat pe suma a doua jocuri random

Am sketchat de asemenea **demonstratia unicitatii** lui Shapley, care implica descompunerea in jocuri unanime si validarea pe acestea.

**Implicatia practica importanta**: orice metoda de explicare a modelelor ML care nu satisface toate 4 axiomele - de exemplu, **feature_importances_** clasic, care nu satisface eficienta - este matematic INFERIOARA fata de SHAP din punct de vedere al garantiilor de corectitudine.

**Urmatorul notebook**: vedem cum se TRADUCE valoarea Shapley in valoarea SHAP, construind ambele explicit pe un model XGBoost simplu cu 3 features pe date sintetice.